#### Elaboração dos gráficos do relatório executivo da FASE 1 - PÓS TECH em Data Analytics

### KPI's

Faturamento por categoria de produto

- Quais categorias concentram a maior parte da receita do e-commerce?  

- As categorias mais vendidas são também as que geram maior faturamento?  

- Qual é o ticket médio por categoria e o que isso revela sobre o valor agregado dos produtos?  

- Existem categorias com alto volume de vendas, mas baixo retorno financeiro?  

- Existem categorias com menor volume, mas maior impacto na receita?  

- O equilíbrio entre volume de vendas e valor agregado está sendo explorado de forma estratégica pelo negócio?  

In [54]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd

#Inicializa a sessão Spark que você vai usar pra ler, tratar e transformar os dados antes de jogar no dashboard.
spark = SparkSession.builder.appName("OlistAnalysis").getOrCreate()

# Base de pedidos: Contém informações sobre o ciclo do pedido, incluindo datas e status (entregue, enviado, cancelado, etc.)
df_orders = spark.read.csv("./archive/olist_orders_dataset.csv", header=True, inferSchema=True, sep=",")

# Base de itens dos pedidos:Contém o detalhe de cada produto vendido, incluindo preço e valor de frete
df_items = spark.read.csv("./archive/olist_order_items_dataset.csv", header=True, inferSchema=True, sep=",")

# Base de clientes:Contém informações geográficas dos clientes (cidade e estado)
df_customers = spark.read.csv("./archive/olist_customers_dataset.csv", header=True, inferSchema=True, sep=",")

# Base de produtos:Contém atributos dos produtos, incluindo a categoria original
df_products = spark.read.csv("./archive/olist_products_dataset.csv", header=True, inferSchema=True, sep=",")

# Base de tradução de categorias:Permite converter os nomes das categorias de produtos para inglês, facilitando a análise
df_translation = spark.read.csv("./archive/product_category_name_translation.csv", header=True, inferSchema=True, sep=",")


In [2]:
# testa se a formatação está correta formatação
display(df_orders)

DataFrame[order_id: string, customer_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp]

In [55]:
# INTEGRAÇÃO DAS BASES DE DADOS
# Realiza o cruzamento das bases para consolidar informações de:
# - pedidos
# - produtos
# - clientes
# - categorias (com tradução)
# O objetivo é criar uma base única para análise de desempenho de produtos

df_produto = (
    df_items
    .join(df_products, on="product_id", how="left")          # adiciona informações do produto
    .join(df_orders, on="order_id", how="left")              # adiciona informações do pedido
    .join(df_customers, on="customer_id", how="left")        # adiciona localização do cliente
    .join(df_translation, on="product_category_name", how="left")  # traduz categorias
)

# SELEÇÃO DAS VARIÁVEIS RELEVANTES
# Mantém apenas as colunas necessárias para a análise de produto,
# focando em faturamento, categoria, localização e data da compra

df_produto = df_produto.select(
    "order_id",
    "product_id",
    "product_category_name",
    "product_category_name_english",
    "price",
    "customer_state",
    "order_purchase_timestamp",
    "order_status"
)

# CRIAÇÃO DE VARIÁVEL TEMPORAL
# Cria uma coluna no formato "ano-mês" para possibilitar análises de crescimento ao longo do tempo

df_produto = df_produto.withColumn(
    "mes",
    F.date_format("order_purchase_timestamp", "yyyy-MM")
)

# Visualização inicial da base tratada
df_produto.show(5)

+--------------------+--------------------+---------------------+-----------------------------+-----+--------------+------------------------+------------+-------+
|            order_id|          product_id|product_category_name|product_category_name_english|price|customer_state|order_purchase_timestamp|order_status|    mes|
+--------------------+--------------------+---------------------+-----------------------------+-----+--------------+------------------------+------------+-------+
|00018f77f2f0320c5...|e5f2d52b802189ee6...|             pet_shop|                     pet_shop|239.9|            SP|     2017-04-26 10:53:06|   delivered|2017-04|
|00042b26cf59d7ce6...|ac6c3623068f30de0...|   ferramentas_jardim|                 garden_tools|199.9|            SP|     2017-02-04 13:57:51|   delivered|2017-02|
|4519eb4d4f3192015...|fde71f25e699ca0a2...|    livros_importados|               books_imported| 61.0|            SP|     2017-08-21 19:07:53|   delivered|2017-08|
|4519e07ee266cdb76...|

In [56]:
#Analisando datas presentes na tabela
df_produto.select("mes") \
    .distinct() \
    .orderBy("mes") \
    .show()

+-------+
|    mes|
+-------+
|2016-09|
|2016-10|
|2016-12|
|2017-01|
|2017-02|
|2017-03|
|2017-04|
|2017-05|
|2017-06|
|2017-07|
|2017-08|
|2017-09|
|2017-10|
|2017-11|
|2017-12|
|2018-01|
|2018-02|
|2018-03|
|2018-04|
|2018-05|
+-------+
only showing top 20 rows


In [57]:
# Tipo das colunas
df_items.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)



In [58]:
# Agrupa os dados por categoria de produto
# Calcula o faturamento total (soma do preço)
# e o volume de vendas (quantidade de produtos vendidos)
# Ordena da maior para a menor categoria em faturamento
df_categoria = (
    df_produto
    .groupBy("product_category_name")
    .agg(
        F.round(F.sum("price"), 2).alias("faturamento"),
        F.count("product_id").alias("volume_vendas")
    )
)


df_categoria.show(20)

+---------------------+-----------+-------------+
|product_category_name|faturamento|volume_vendas|
+---------------------+-----------+-------------+
|                  pcs|  222963.13|          203|
|                bebes|  411764.89|         3065|
|                artes|   24202.64|          209|
|            cine_foto|    6933.46|           72|
|     moveis_decoracao|  729762.49|         8334|
|             pc_gamer|    1545.95|            9|
| construcao_ferram...|  144677.59|          929|
| tablets_impressao...|    7528.41|           83|
|    artigos_de_festas|    4485.18|           43|
| fashion_roupa_mas...|   10797.82|          132|
|     artigos_de_natal|    8800.82|          153|
|           la_cuisine|    2054.99|           14|
|               flores|    1110.04|           33|
|      livros_tecnicos|   19096.06|          267|
|                 NULL|  179535.28|         1603|
|       telefonia_fixa|    59583.0|          264|
| construcao_ferram...|   40544.52|          194|


In [59]:
# Correção de precição de float
df_categoria = df_categoria.withColumn(
    "ticket_medio",
    F.round(F.col("faturamento") / F.col("volume_vendas"), 2)
)

In [ ]:
#Definindo valor medio por categoria de produto
df_categoria = df_categoria.withColumn(
    "ticket_medio",
    F.round(F.col("faturamento") / F.col("volume_vendas"), 2)
)

#Ordenando as categorias por faturamento
df_categoria = df_categoria.orderBy(F.desc("faturamento"))


In [62]:
# EXPORTAÇÃO DA BASE TRATADA
df_pandas = df_categoria.toPandas()
df_pandas.to_csv("minha_base.csv", index=False)